# Data Aggregation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ixnix/GE5280PythonGeosciences/blob/main/colab/module_9/9_DataAggregation.ipynb)

*Run this notebook on Google Colab — no setup required.*

# Module 9: Data Aggregation


In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import Image

import warnings
warnings.filterwarnings('ignore')

## Introduction
Data aggregation and grouping are essential techniques for summarizing and analyzing large datasets. In pandas, the groupby() method implements the split–apply–combine strategy, allowing you to split data into groups, apply functions to each group, and combine the results into a single output. These tools are especially useful for computing summary statistics, reshaping data, and performing operations across categories without manual iteration.

## Learning Objectives
- Perform data aggregation and group operations in pandas using built-in and custom functions.
- Apply the split–apply–combine workflow to summarize, transform, and filter grouped data.
- Use custom aggregation functions to solve complex analytical problems beyond simple descriptive statistics.

## Reading
- [Chapter 10](https://wesmckinney.com/book/data-aggregation)

Categorizing a dataset and applying a function to each group, whether an aggregation or transformation, is often a critical component of a data analysis workflow. After loading, merging, and preparing a dataset, you may need to compute group statistics or possibly pivot tables for reporting or visualization purposes. pandas provides a flexible groupby interface, enabling you to slice, dice, and summarize datasets in a natural way.

Groupby operations follow the **split-apply-combine** concept:
- The split step involves breaking up and grouping a DataFrame depending on the value of the specified key.
- The apply step involves computing some function, usually an aggregate, transformation, or filtering, within the individual groups.
- The combine step merges the results of these operations into an output array.


In [ ]:
Image(filename='https://raw.githubusercontent.com/ixnix/GE5280PythonGeosciences/main/module_9/img/split-apply-combine.jpg',width=400)

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/ixnix/GE5280PythonGeosciences/main/module_9/data/GVP_Volcano_List_Holocene.csv',header=1,dtype={'Volcano Number':str})
df

Recall that we added a new column for the eruption date, based on the existing column 'Last Known Eruption'.

In [ ]:
# First, drop all the rows with unknown eruption date
df = df[~df['Last Known Eruption'].str.contains('Unknown')] 

# put in the 'Date' column to keep the numeric part of 'Last Known Eruption' column
# .str.split() produces a string, which is converted to integer
df['Date'] = df['Last Known Eruption'].str.split(' ',expand=True)[0].astype('int')

# put in the 'CE/BCE' column:
df['CE/BCE'] = df['Last Known Eruption'].str.split(' ',expand=True)[1].astype('str')

# update the 'Date' column based on CE/BCE
df.loc[df['CE/BCE'].str.contains('BCE'), 'Date'] = -1*df['Date']

df.head()

Suppose you want to count the number of volcanoes in each Region. Conceptually, you would take these steps:
- split the DataFrame into groups, each with all the rows corresponding to the same Region
- count the size of each group
- gather the results for each group together

In pandas, this is done via the **groupby** operation

In [ ]:
grps = df.groupby('Region')
type(grps)

Notice that what is returned is not a set of DataFrames, but a **DataFrameGroupBy** object, which essentially describes how the rows of the original dataset have been split. There are some attributes and methods available for us to access groups information.

This object is where the magic is: you can think of it as a special view of the DataFrame, which is poised to dig into the groups but does no actual computation until the aggregation is applied.

In [ ]:
# use ngroups attribute to get the number of groups
grps.ngroups

In [ ]:
# Use groups attribute to get groups object. The integer numbers in the list are the row number for each group.
grps.groups

In [ ]:
#use size() method to compute and display group sizes.
grps.size()

In [ ]:
# To preview the groups, we can call first() or last() to preview the result with the first or last entry.
grps.first()

**.groupby( )** groups things in your DataFrame by unique values in a **Series**, for example grouping everything by 'Region'. 

Then, **.describe( )** can be used to summarize some useful statistics of each group.

In [ ]:
grps['Primary Volcano Type'].describe()

This tells us that, for example, out of 59 volcanoes in 'Alaska', 40 are 'stratovolcanoes'.

Let's go back to our question: what is the number of volcanoes of each region?

You can use the .count() method on the 'Region' column of the groupby object. It returns a pandas **Series** with the list of regions as the index.

In [ ]:
df.groupby('Region')['Region'].count()

For visualization we can use a bar plot. Note that Series or DataFrame have built-in plot function, which is a wrapper of the matplotlib plot function.

In [ ]:
dfg = df.groupby('Region')['Region'].count()
ax = dfg.plot(kind='bar',width=.7,figsize=(6,4))
ax.set_ylabel('Count')

In [ ]:
# We can do some customizations.

dfg = df.groupby('Region')['Region'].count()
ax = dfg.plot(kind='bar',width=.7,figsize=(6,4),color='gray')
ax.set_ylabel('Count')
ax.set_xticks([])

for i, idx in enumerate(dfg.index):
    ax.text(i,0,idx,va='bottom',c='k',ha='center',rotation=90,size='small')
    
plt.show()

What if we want to find out the highest volcano in each region?

In [ ]:
df.groupby('Region')['Elevation (m)'].max()

When is the most recent eruption in each country?

In [ ]:
df.groupby('Country')['Date'].max()

The above methods of groupby objects, namely count() and max(), are called **aggregation** statistics. 

There are a list of aggregation methods:
    
|Function name| Description|
|-------------|------------|
|count| Number of non-NA values in the group|
|sum| Sum of non-NA values|
|mean| Mean of non-NA values|
|median| Arithmetic median of non-NA values|
|std, var| Unbiased (n – 1 denominator) standard deviation and variance|
|min, max| Minimum and maximum of non-NA values|
|prod| Product of non-NA values|
|first, last| First and last non-NA values|

You can also define your own function and use as aggregation method. For instance, what is the height difference between the lowest and highest volcano within each region?

To use your own aggregation functions, pass any function that aggregates an array to the aggregate or **agg** method:

In [ ]:
def max_minus_min(arr):
    return arr.max()-arr.min()

df.groupby('Region')['Elevation (m)'].agg(max_minus_min)

To find the most common rock type of each region:

In [ ]:
def most_common(arr):
    return arr.value_counts().index[0]

df.groupby('Region')['Dominant Rock Type'].agg(most_common)

In the above, the **value_counts()** method is applied to the 'Dominant Rock Type' column of each group, and returns a **Series** containing counts of unique values. 

The Series is in descending order so that the first element is the most frequently-occurring element. For example,

In [ ]:
df['Dominant Rock Type'].value_counts()

In the above, the aggregation was performed on a specified column. Sometimes it is desirable to include all columns from the DataFrame.

In [ ]:
def most_common(arr):
    return arr.value_counts().index[0]

df.groupby('Region').agg(most_common)

You can also pass a list of functions to the agg method. It will return a **DataFrame** with column names taken from the functions.

In [ ]:
df.groupby('Region')['Elevation (m)'].agg(['max','min',max_minus_min])

You can specify your own column names for the output.

In [ ]:
df.groupby('Region')['Elevation (m)'].agg(
    ELV_max='max',
    ELV_min='min',
    ELV_range=max_minus_min
)

What if we want to calculate different statistics depending on the column, for example, the most recent eruption date and the average elevation within each region?

In [ ]:
df.groupby('Region')[['Date','Elevation (m)']].agg({
    'Date':'max',
    'Elevation (m)':'mean',
    })

In all of the examples above, the aggregated data comes back with an index composed from the unique group key combinations. 

You can disable this behavior in most cases by passing as_index=False to groupby.

In [ ]:
df.groupby('Region',as_index=False)[['Date','Elevation (m)']].agg({
    'Date':'max',
    'Elevation (m)':'mean',
    })

You can also groupby on multiple columns:

In [ ]:
df.groupby(['Region','Primary Volcano Type'])[['Date','Elevation (m)']].agg({
    'Date':'max',
    'Elevation (m)':'mean',
    })

The above examples use a single or multiple columns of the DataFrame itself as the grouping key. However, the grouping key can take many forms, and the keys do not have to be all of the same type:
- A list or array of values that is the same length as the axis being grouped
- A value indicating a column name in a DataFrame
- A dict or Series giving a correspondence between the values on the axis being grouped and the group names
- A function to be invoked on the axis index or the individual labels in the index

For example, you want to count the number of volcanoes for every 10-degree longitude bands starting from -180 to 180 degrees.

In [ ]:
binEdges = np.arange(-180,190,10)
bins = pd.cut(df['Longitude'],binEdges) # cut df into different bins by longitude
Nlon = df['Longitude'].groupby(bins).count()
Nlon.name = 'Count'
Nlon

The **pd.cut()** function is used to cut Series or DataFrame when you need to segment and sort data values into bins. This function is also useful for going from a continuous variable to a categorical variable. For example, cut could convert ages to groups of age ranges. Supports binning into an equal number of bins, or a pre-specified array of bins.

More info: https://pandas.pydata.org/docs/reference/api/pandas.cut.html

In [ ]:
ax = Nlon.plot(kind='barh',width=.6,color='k',figsize=(4,8))
ax.set_xlabel('Count')